In [3]:
# %% [markdown]
# # 11 — Prospect Profiler Model
#
# ## Objective
# Build the first prospect profiling layer from the synthetic survey base.
#
# This notebook will:
# - load prospect survey data
# - map survey variables into a risk-style scoring framework
# - generate a predicted archetype proxy
# - estimate a first prospect risk score
#
# Main output:
# - `prospect_profile_scores.csv`

# %%
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

# %%
import pandas as pd
import numpy as np

from src.config import PROCESSED_DIR, OUTPUTS_DIR
from src.data.load_data import load_prospect_survey, load_insured_members, load_policies

# %% [markdown]
# ## Load data

# %%
prospects = load_prospect_survey()
members = load_insured_members()

member_master_path = PROCESSED_DIR / "member_master.csv"
if member_master_path.exists():
    pricing_reference = pd.read_csv(member_master_path)
else:
    pricing_reference = load_policies()

print("prospects:", prospects.shape)
print("members:", members.shape)
print("pricing_reference:", pricing_reference.shape)

display(prospects.head(3))
print(prospects.columns.tolist())

# %% [markdown]
# ## Inspect prospect columns

# %%
prospects.dtypes.reset_index().rename(columns={"index": "column", 0: "dtype"})

# %% [markdown]
# ## Helper functions

# %%
def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce")

def normalize_01(series):
    s = safe_numeric(series).copy()
    if s.notna().sum() == 0:
        return pd.Series(np.nan, index=series.index)
    mn, mx = s.min(), s.max()
    if pd.isna(mn) or pd.isna(mx) or mn == mx:
        return pd.Series(0.5, index=series.index)
    return (s - mn) / (mx - mn)

def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def yes_no_to_int(series):
    return (
        series.astype(str)
        .str.strip()
        .str.lower()
        .isin(["1", "true", "yes", "y", "si", "sí"])
        .astype(int)
    )

# %% [markdown]
# ## Detect candidate survey fields

# %%
age_col = first_existing(prospects, ["age", "prospect_age"])
sex_col = first_existing(prospects, ["sex", "gender"])
dependents_col = first_existing(prospects, ["dependents_n", "dependents", "family_dependents"])
smoker_col = first_existing(prospects, ["smoker_flag", "smoking_flag", "smoker"])
bmi_col = first_existing(prospects, ["bmi_group", "bmi_category"])
chronic_col = first_existing(prospects, ["chronic_condition_flag", "chronic_flag"])
price_col = first_existing(prospects, ["price_sensitivity", "price_preference", "budget_sensitivity"])
coverage_col = first_existing(prospects, ["coverage_preference", "coverage_priority"])
network_col = first_existing(prospects, ["network_preference", "network_priority"])
plan_pref_col = first_existing(prospects, ["preferred_plan_type", "plan_preference"])
activity_col = first_existing(prospects, ["physical_activity_level", "activity_level"])
id_col = first_existing(prospects, ["prospect_id", "id"])

detected = {
    "id_col": id_col,
    "age_col": age_col,
    "sex_col": sex_col,
    "dependents_col": dependents_col,
    "smoker_col": smoker_col,
    "bmi_col": bmi_col,
    "chronic_col": chronic_col,
    "price_col": price_col,
    "coverage_col": coverage_col,
    "network_col": network_col,
    "plan_pref_col": plan_pref_col,
    "activity_col": activity_col,
}

pd.DataFrame(detected.items(), columns=["field", "detected_column"])

# %% [markdown]
# ## Create prospect profiling base

# %%
prospect_profile = prospects.copy()

if id_col is None:
    prospect_profile["prospect_id"] = np.arange(1, len(prospect_profile) + 1)
    id_col = "prospect_id"

# numeric conversions when relevant
for col in [age_col, dependents_col, price_col]:
    if col is not None:
        prospect_profile[col] = pd.to_numeric(prospect_profile[col], errors="coerce")

# %% [markdown]
# ## Risk proxy scoring rules

# %%
prospect_profile["risk_points"] = 0

if age_col is not None:
    prospect_profile["risk_points"] += np.where(prospect_profile[age_col] >= 60, 2, 0)
    prospect_profile["risk_points"] += np.where(
        (prospect_profile[age_col] >= 45) & (prospect_profile[age_col] < 60), 1, 0
    )

if dependents_col is not None:
    prospect_profile["risk_points"] += np.where(prospect_profile[dependents_col] >= 3, 1, 0)

if smoker_col is not None:
    prospect_profile["risk_points"] += yes_no_to_int(prospect_profile[smoker_col])

if chronic_col is not None:
    prospect_profile["risk_points"] += 2 * yes_no_to_int(prospect_profile[chronic_col])

if bmi_col is not None:
    bmi_risk = prospect_profile[bmi_col].astype(str).str.lower().isin(
        ["obese", "obesity", "overweight"]
    ).astype(int)
    prospect_profile["risk_points"] += bmi_risk

if activity_col is not None:
    low_activity = prospect_profile[activity_col].astype(str).str.lower().isin(
        ["low", "sedentary"]
    ).astype(int)
    prospect_profile["risk_points"] += low_activity

risk_points_max = prospect_profile["risk_points"].max() if prospect_profile["risk_points"].notna().sum() > 0 else 1
if risk_points_max == 0:
    risk_points_max = 1

prospect_profile["prospect_risk_score"] = prospect_profile["risk_points"] / risk_points_max

prospect_profile["prospect_risk_segment"] = pd.cut(
    prospect_profile["prospect_risk_score"],
    bins=[-0.01, 0.20, 0.40, 0.60, 0.80, 1.01],
    labels=["very_low", "low", "medium", "high", "very_high"]
)

display(
    prospect_profile[[
        c for c in [
            id_col,
            age_col,
            dependents_col,
            smoker_col,
            chronic_col,
            bmi_col,
            activity_col,
            "risk_points",
            "prospect_risk_score",
            "prospect_risk_segment",
        ] if c is not None and c in prospect_profile.columns
    ]].head(10)
)

# %% [markdown]
# ## Archetype proxy assignment

# %%
def assign_archetype(row):
    risk = row["prospect_risk_score"]

    price_pref = str(row.get(price_col, "")).lower() if price_col is not None else ""
    coverage_pref = str(row.get(coverage_col, "")).lower() if coverage_col is not None else ""
    network_pref = str(row.get(network_col, "")).lower() if network_col is not None else ""

    if risk >= 0.75 and ("high" in coverage_pref or "broad" in network_pref or "full" in coverage_pref):
        return "high_need_protected"
    elif risk >= 0.50:
        return "managed_risk"
    elif "high" in price_pref or "sensitive" in price_pref or "budget" in price_pref:
        return "price_sensitive_basic"
    else:
        return "balanced_standard"

prospect_profile["predicted_archetype"] = prospect_profile.apply(assign_archetype, axis=1)

prospect_profile["predicted_archetype"].value_counts(dropna=False).rename_axis("predicted_archetype").reset_index(name="count")

# %% [markdown]
# ## Expected utilization proxy

# %%
prospect_profile["expected_utilization_proxy"] = 0.50 * prospect_profile["prospect_risk_score"]

if dependents_col is not None:
    dep_norm = normalize_01(prospect_profile[dependents_col]).fillna(0)
    prospect_profile["expected_utilization_proxy"] += 0.20 * dep_norm

if chronic_col is not None:
    chronic_norm = yes_no_to_int(prospect_profile[chronic_col])
    prospect_profile["expected_utilization_proxy"] += 0.20 * chronic_norm

if age_col is not None:
    age_norm = normalize_01(prospect_profile[age_col]).fillna(0)
    prospect_profile["expected_utilization_proxy"] += 0.10 * age_norm

prospect_profile["expected_utilization_proxy"] = prospect_profile["expected_utilization_proxy"].clip(0, 1)

# %% [markdown]
# ## Estimated premium proxy
#
# We calibrate a rough premium band using existing premium distributions.

# %%
premium_reference_col = None
for candidate in ["premium_monthly", "premium_annual"]:
    if candidate in pricing_reference.columns:
        premium_reference_col = candidate
        break

print("Detected premium reference column:", premium_reference_col)

if premium_reference_col is None:
    member_premium_mean = 100
    member_premium_std = 20
else:
    premium_series = pd.to_numeric(pricing_reference[premium_reference_col], errors="coerce")

    if premium_reference_col == "premium_annual":
        premium_series = premium_series / 12

    member_premium_mean = premium_series.mean()
    member_premium_std = premium_series.std()

if pd.isna(member_premium_mean):
    member_premium_mean = 100
if pd.isna(member_premium_std):
    member_premium_std = 20

prospect_profile["estimated_monthly_premium_mid"] = (
    member_premium_mean * (0.75 + prospect_profile["prospect_risk_score"])
)

prospect_profile["estimated_monthly_premium_low"] = prospect_profile["estimated_monthly_premium_mid"] * 0.90
prospect_profile["estimated_monthly_premium_high"] = prospect_profile["estimated_monthly_premium_mid"] * 1.10

display(
    prospect_profile[[
        c for c in [
            id_col,
            "prospect_risk_score",
            "prospect_risk_segment",
            "predicted_archetype",
            "expected_utilization_proxy",
            "estimated_monthly_premium_low",
            "estimated_monthly_premium_mid",
            "estimated_monthly_premium_high",
        ] if c in prospect_profile.columns
    ]].head(10)
)

# %% [markdown]
# ## Build final output

# %%
output_cols = [c for c in [
    id_col,
    age_col,
    sex_col,
    dependents_col,
    smoker_col,
    bmi_col,
    chronic_col,
    price_col,
    coverage_col,
    network_col,
    plan_pref_col,
    "risk_points",
    "prospect_risk_score",
    "prospect_risk_segment",
    "predicted_archetype",
    "expected_utilization_proxy",
    "estimated_monthly_premium_low",
    "estimated_monthly_premium_mid",
    "estimated_monthly_premium_high",
] if c is not None and c in prospect_profile.columns]

prospect_profile_scores = prospect_profile[output_cols].copy()
print("prospect_profile_scores:", prospect_profile_scores.shape)
display(prospect_profile_scores.head(10))

# %% [markdown]
# ## Save output

# %%
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

output_file = OUTPUTS_DIR / "prospect_profile_scores.csv"
prospect_profile_scores.to_csv(output_file, index=False)

print("Saved:", output_file)

# %% [markdown]
# ## Final notes

# %%
notes = [
    "This is the first survey-to-risk translation layer for prospects.",
    "The premium calibration now uses member_master.csv or policies.csv as pricing reference.",
    "The logic is intentionally simple and explainable for dashboard demo use.",
]

for i, note in enumerate(notes, 1):
    print(f"{i}. {note}")

PROJECT_ROOT: C:\Users\dolivares\Desktop\novares-securehealth
prospects: (7000, 30)
members: (4000, 32)
pricing_reference: (4000, 63)


,prospect_id,age,age_band,sex,region,dependents_n,self_rated_health,chronic_condition_flag,chronic_condition_count,recurrent_medication_flag,...,maternity_interest_flag,pharmacy_need_flag,chronic_program_interest_flag,predicted_ground_truth_archetype,risk_tier,expected_utilization_band,recommended_plan,recommended_plan_tier,notes_driver_1,notes_driver_2
0,PRS000001,30,25-34,M,Alto Paraná,1,Good,0,0,0,...,0,0,0,Family Planner,Moderate,Medium,Family Plus,Mid,Family/dependent complexity,Preference for broader network
1,PRS000002,47,45-54,F,Central,1,Good,0,0,0,...,0,0,0,Preventive User,Moderate,Low,Standard,Mid,Moderate preventive utilization,Balanced coverage preference
2,PRS000003,18,18-24,F,Asunción,1,Excellent,0,0,0,...,0,0,0,Healthy Low User,Low,Very Low,Essential,Basic,Low recent utilization,Price-sensitive coverage preference


['prospect_id', 'age', 'age_band', 'sex', 'region', 'dependents_n', 'self_rated_health', 'chronic_condition_flag', 'chronic_condition_count', 'recurrent_medication_flag', 'visits_12m_band', 'er_visits_12m_band', 'hospitalization_24m_flag', 'smoker_flag', 'bmi_group', 'physical_activity_level', 'preventive_mindset', 'price_vs_coverage_preference', 'copay_tolerance', 'network_preference', 'maternity_interest_flag', 'pharmacy_need_flag', 'chronic_program_interest_flag', 'predicted_ground_truth_archetype', 'risk_tier', 'expected_utilization_band', 'recommended_plan', 'recommended_plan_tier', 'notes_driver_1', 'notes_driver_2']


,prospect_id,age,dependents_n,smoker_flag,chronic_condition_flag,bmi_group,physical_activity_level,risk_points,prospect_risk_score,prospect_risk_segment
0,PRS000001,30,1,0,0,Normal,Medium,0,0.000000,very_low
1,PRS000002,47,1,0,0,Normal,Medium,1,0.142857,very_low
2,PRS000003,18,1,0,0,Overweight,High,1,0.142857,very_low
3,PRS000004,48,2,0,0,Overweight,Medium,2,0.285714,low
4,PRS000005,29,2,1,0,Normal,Medium,1,0.142857,very_low
5,PRS000006,48,1,1,1,Overweight,Low,6,0.857143,very_high
6,PRS000007,42,1,0,0,Normal,Medium,0,0.000000,very_low
7,PRS000008,43,2,0,1,Obese,High,3,0.428571,medium
8,PRS000009,39,0,0,0,Overweight,Medium,1,0.142857,very_low
9,PRS000010,34,0,0,0,Overweight,Medium,1,0.142857,very_low


Detected premium reference column: premium_monthly


,prospect_id,prospect_risk_score,prospect_risk_segment,predicted_archetype,expected_utilization_proxy,estimated_monthly_premium_low,estimated_monthly_premium_mid,estimated_monthly_premium_high
0,PRS000001,0.000000,very_low,balanced_standard,0.069355,258.339280,287.043645,315.748009
1,PRS000002,0.142857,very_low,balanced_standard,0.168203,307.546762,341.718625,375.890488
2,PRS000003,0.142857,very_low,balanced_standard,0.121429,307.546762,341.718625,375.890488
3,PRS000004,0.285714,low,balanced_standard,0.291244,356.754244,396.393605,436.032965
4,PRS000005,0.142857,very_low,balanced_standard,0.189171,307.546762,341.718625,375.890488
5,PRS000006,0.857143,very_high,high_need_protected,0.726959,553.584173,615.093525,676.602878
6,PRS000007,0.000000,very_low,balanced_standard,0.088710,258.339280,287.043645,315.748009
7,PRS000008,0.428571,medium,balanced_standard,0.554608,405.961726,451.068585,496.175444
8,PRS000009,0.142857,very_low,balanced_standard,0.105300,307.546762,341.718625,375.890488
9,PRS000010,0.142857,very_low,balanced_standard,0.097235,307.546762,341.718625,375.890488


prospect_profile_scores: (7000, 16)


,prospect_id,age,sex,dependents_n,smoker_flag,bmi_group,chronic_condition_flag,network_preference,risk_points,prospect_risk_score,prospect_risk_segment,predicted_archetype,expected_utilization_proxy,estimated_monthly_premium_low,estimated_monthly_premium_mid,estimated_monthly_premium_high
0,PRS000001,30,M,1,0,Normal,0,Broad,0,0.000000,very_low,balanced_standard,0.069355,258.339280,287.043645,315.748009
1,PRS000002,47,F,1,0,Normal,0,Narrow,1,0.142857,very_low,balanced_standard,0.168203,307.546762,341.718625,375.890488
2,PRS000003,18,F,1,0,Overweight,0,Balanced,1,0.142857,very_low,balanced_standard,0.121429,307.546762,341.718625,375.890488
3,PRS000004,48,F,2,0,Overweight,0,Balanced,2,0.285714,low,balanced_standard,0.291244,356.754244,396.393605,436.032965
4,PRS000005,29,F,2,1,Normal,0,Balanced,1,0.142857,very_low,balanced_standard,0.189171,307.546762,341.718625,375.890488
5,PRS000006,48,M,1,1,Overweight,1,Broad,6,0.857143,very_high,high_need_protected,0.726959,553.584173,615.093525,676.602878
6,PRS000007,42,M,1,0,Normal,0,Balanced,0,0.000000,very_low,balanced_standard,0.088710,258.339280,287.043645,315.748009
7,PRS000008,43,M,2,0,Obese,1,Narrow,3,0.428571,medium,balanced_standard,0.554608,405.961726,451.068585,496.175444
8,PRS000009,39,F,0,0,Overweight,0,Broad,1,0.142857,very_low,balanced_standard,0.105300,307.546762,341.718625,375.890488
9,PRS000010,34,F,0,0,Overweight,0,Balanced,1,0.142857,very_low,balanced_standard,0.097235,307.546762,341.718625,375.890488


Saved: C:\Users\dolivares\Desktop\novares-securehealth\data\outputs\prospect_profile_scores.csv
1. This is the first survey-to-risk translation layer for prospects.
2. The premium calibration now uses member_master.csv or policies.csv as pricing reference.
3. The logic is intentionally simple and explainable for dashboard demo use.
